In [ ]:
import pandas as pd
borough= pd.read_csv(r"C:\Jason\Ride share project\taxi_zone_lookup.csv")
print(borough.shape)
cols = ['hvfhs_license_num',
        'request_datetime', 
        'trip_miles',
        'trip_time', #in seconds
        'PULocationID', 
        'DOLocationID',
        'congestion_surcharge', 
        'shared_request_flag']
uber_df = pd.read_parquet(r"C:\Jason\Ride share project\fhvhv_tripdata_2025-01.parquet", columns=cols)
uber_df = uber_df[uber_df['hvfhs_license_num'] == 'HV0003'].copy()
print(uber_df.shape)
uber_df.head()


In [ ]:
print(borough.columns.tolist())
print(borough.head())


In [ ]:
#merge pickup borough
uber_df = uber_df.merge(borough[['LocationID', 'Borough', 'Zone']], 
                        left_on='PULocationID', right_on='LocationID', how='left')
uber_df.rename(columns={'Borough': 'PU_Borough', 
                        'Zone': 'PU_Zone', 
                        'LocationID': 'PU_LocationID'}, 
               inplace=True)
#merge dropoff locations
uber_df = uber_df.merge(borough[['LocationID', 'Borough', 'Zone']], 
                        left_on='DOLocationID', right_on='LocationID', how='left')
uber_df.rename(columns={'Borough': 'DO_Borough', 
                        'Zone': 'DO_Zone', 
                        'LocationID': 'DO_LocationID'}, 
               inplace=True)

In [ ]:
uber_df = uber_df[(uber_df['trip_miles']>0.1)
                   & (uber_df['trip_time']>60)&
                      (uber_df['trip_miles']<100)].copy()
uber_df.head()

In [ ]:
#Time Bucket, day of week, weekend flag
#drives at different times of the day will have different patterns, so we can bucket the time into bands (e.g., morning, afternoon, evening) and also identify if it's a weekend or not.
uber_df['hour'] = uber_df['request_datetime'].dt.hour
uber_df['day_of_week'] = uber_df['request_datetime'].dt.dayofweek
uber_df['is_weekend'] = uber_df['day_of_week'].isin([4,5,6]).astype(int)

#bands
def time_band(hour):
    if 0 <= hour < 6:
        return 'Late Night'
    elif 6 <= hour < 10:
        return 'Morning'
    elif 10 <= hour < 16:
        return 'Midday'
    elif 16 <= hour < 20:
        return 'Evening'
    else:
        return 'Night'
uber_df['time_band'] = uber_df['hour'].apply(time_band)
print(uber_df[['hour', 'time_band']].head())

In [ ]:
uber_df['avg_speed_mph'] = uber_df['trip_miles'] / (uber_df['trip_time'] / 3600)#average speed in miles per hour
uber_df['trip_time_minutes'] = uber_df['trip_time'] / 60#trip time in minutes

In [ ]:
uber_df['exposure'] = uber_df['trip_miles'] #how far the drive is
uber_df['avg_speed_mph'] = uber_df['trip_miles'] / (uber_df['trip_time'] / 3600) #average speed in miles per hour
uber_df['highway_flag'] = (uber_df['avg_speed_mph'] > 45).astype(int) #assuming trips with avg speed > 45 mph are likely on highways

In [ ]:
#trip value by time buckets
time_stats = uber_df.groupby('time_band').agg(
    trip_count = ('trip_miles', 'count'),
    avg_miles = ('trip_miles', 'mean'),
    total_miles = ('trip_miles', 'sum'),
    avg_speed = ('avg_speed_mph', 'mean')).reset_index()
time_stats['pct_of_trips'] = time_stats['trip_count'] / time_stats['trip_count'].sum() * 100
print(time_stats.round(2))                                       

In [ ]:
#trips between boroughs to see which routes are common, which are more likely to be taken etc.
borough_stats = uber_df.groupby(['PU_Borough', 'DO_Borough']).agg(
    trip_count = ('trip_miles', 'count'),
    avg_miles = ('trip_miles', 'mean'),
    total_miles = ('trip_miles', 'sum'),
    avg_speed = ('avg_speed_mph', 'mean')).reset_index()
borough_stats['pct_of_trips'] = borough_stats['trip_count'] / borough_stats['trip_count'].sum() * 100
borough_stats = borough_stats.sort_values('trip_count', ascending=False)

print(borough_stats[['PU_Borough', 'DO_Borough', 'trip_count', 'pct_of_trips']].round(2))


In [ ]:
#highway marker
speed_stats = uber_df.groupby('highway_flag').agg(
    trip_count = ('trip_miles', 'count'),
    avg_miles = ('trip_miles', 'mean'),
    total_miles = ('trip_miles', 'sum'),
    avg_speed = ('avg_speed_mph', 'mean'),).reset_index()
    
speed_stats['highway_flag'] = speed_stats['highway_flag'].map({0: 'city', 1: 'highway'})
print(speed_stats.round(2))

In [ ]:
print(uber_df.columns.tolist())
print(borough_stats.columns.tolist())

In [ ]:

import seaborn as sns
import matplotlib.pyplot as plt
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(2,2, figsize=(15,10))

sns.countplot(data=uber_df, x='time_band', ax=axes[0,0])
axes[0,0].set_title('Trip count by time band')
axes[0,0].set_xlabel('Time band')
axes[0,0].set_ylabel('Trip count')
